# Week 19, Day 4: Python's sort: Timsort

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w019d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from functools import cmp_to_key

comparisons = 0
def compare(a, b):
    global comparisons
    comparisons += 1
    return (a > b) - (a < b)        # -1, 0 or 1

sorted(list(range(1000)), key=cmp_to_key(compare))
print(comparisons)

In [ ]:
def find_runs(items):
    runs = []                     # (start, end) positions, end not included
    i, n = 0, len(items)
    while i < n:
        j = i + 1
        if j < n and items[j] < items[i]:              # strictly falling
            while j < n and items[j] < items[j - 1]:
                j += 1
            items[i:j] = items[i:j][::-1]              # flip it to rising
        else:                                           # rising (or one item)
            while j < n and items[j] >= items[j - 1]:
                j += 1
        runs.append((i, j))
        i = j
    return runs

data = [3, 5, 9, 12, 8, 6, 2, 7, 7, 10, 1]
print(find_runs(data), data)

In [ ]:
from bisect import bisect_right

def binary_insertion_sort(items, lo, hi, start):
    """items[lo:start] is already sorted; extend the sorted part to items[lo:hi]."""
    for i in range(start, hi):
        x = items[i]
        pos = bisect_right(items, x, lo, i)    # after any equal items: stable
        items[pos + 1:i + 1] = items[pos:i]    # shift the bigger items right
        items[pos] = x

data = [2, 5, 9, 1, 7, 3, 8]
binary_insertion_sort(data, 0, len(data), 3)   # [2, 5, 9] is a run; add the rest
print(data)

In [ ]:
def merge(a, b):
    out, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if b[j] < a[i]:              # take from b only if strictly smaller: stable
            out.append(b[j]); j += 1
        else:
            out.append(a[i]); i += 1
    return out + a[i:] + b[j:]

def simple_timsort(items, minrun=32):
    items = items[:]
    n = len(items)
    runs, i = [], 0
    while i < n:
        j = i + 1
        if j < n and items[j] < items[i]:
            while j < n and items[j] < items[j - 1]:
                j += 1
            items[i:j] = items[i:j][::-1]
        else:
            while j < n and items[j] >= items[j - 1]:
                j += 1
        end = min(max(j, i + minrun), n)          # extend short runs to minrun
        for k in range(j, end):                    # insertion sort the extra items in
            x, m = items[k], k - 1
            while m >= i and items[m] > x:
                items[m + 1] = items[m]
                m -= 1
            items[m + 1] = x
        runs.append(items[i:end])
        i = end
    while len(runs) > 1:                           # merge neighbours, round after round
        runs = [merge(runs[k], runs[k + 1]) if k + 1 < len(runs) else runs[k] for k in range(0, len(runs), 2)]
    return runs[0] if runs else []

import random
data = [random.randint(0, 999) for _ in range(5000)]
print(simple_timsort(data) == sorted(data))

> **Key idea.** Timsort cuts the list into runs (stretches already in order, reversing falling ones), extends short runs to about 32 to 64 items with binary insertion sort, and merges neighbouring runs in a balanced way. O(n) on sorted or reversed data, O(n log n) worst case, stable.

In [ ]:
from functools import cmp_to_key
import random, math

def count_comparisons(data):
    count = 0
    def compare(a, b):
        nonlocal count
        count += 1
        return (a > b) - (a < b)
    sorted(data, key=cmp_to_key(compare))
    return count

n = 10_000
rng = random.Random(1)
cases = {
    "sorted": list(range(n)),
    "reversed": list(range(n, 0, -1)),
    "two sorted halves": list(range(0, n, 2)) + list(range(1, n, 2)),
    "sorted + 10 random at end": list(range(n - 10)) + [rng.randint(0, n) for _ in range(10)],
    "random": rng.sample(range(n), n),
}
for name, data in cases.items():
    print(f"{name:<28}{count_comparisons(data):>8}")
print(f"{'n log2 n':<28}{round(n * math.log2(n)):>8}")

### Exercise: Find the runs

Write `runs(items)` returning a list of the runs as lists, cutting the list wherever the order goes **down** (each run is non-decreasing). Don't reverse anything for this one.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert runs([3, 5, 9, 12, 8, 6, 7, 7, 10, 1]) == [[3, 5, 9, 12], [8], [6, 7, 7, 10], [1]]\n    assert runs([]) == [] and runs([4]) == [[4]] and runs([1, 2, 3]) == [[1, 2, 3]]\n    assert len(runs(list(range(100, 0, -1)))) == 100")

### Exercise: Rising and falling runs

Write `timsort_runs(items)` that works on a copy and returns the runs as lists, the way Timsort finds them: a run that goes **strictly down** is collected as far as it goes and then reversed; otherwise the run goes up (non-decreasing) as far as it can. Every returned run is then non-decreasing.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert timsort_runs([3, 5, 9, 12, 8, 6, 2, 7, 7, 10, 1]) == [[3, 5, 9, 12], [2, 6, 8], [7, 7, 10], [1]]\n    assert timsort_runs(list(range(10, 0, -1))) == [list(range(1, 11))]\n    assert timsort_runs([]) == [] and timsort_runs([5, 5, 4]) == [[5, 5], [4]]\n    assert timsort_runs([2, 2, 1]) == [[2, 2], [1]]")

### Exercise: Binary insertion sort

Write `binary_insertion_sort(items)` that sorts a list in place, stably, finding each item's place with your **own** binary search (the first position whose item is bigger than it, among the sorted part), then shifting the bigger items right. Return the number of comparisons your binary searches made.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    for data in ([5, 2, 9, 1, 7], [], [1], [2, 1], [3, 3, 1]):\n        copy = list(data)\n        binary_insertion_sort(copy)\n        assert copy == sorted(data)\n    recs = [(random.Random(i).randint(0, 3), i) for i in range(200)]\n    keys = [r[0] for r in recs]\n    copy = keys[:]\n    binary_insertion_sort(copy)\n    assert copy == sorted(keys)\n    c = binary_insertion_sort(random.Random(1).sample(range(1000), 1000))\n    assert c < 10_000\n    assert \"bisect\" not in source")

### Exercise: Count the comparisons of sorted

Write `comparisons_of_sorted(data)` that returns how many comparisons `sorted(data)` makes, using `cmp_to_key`. Then set `sorted_1000`, `reversed_1000` and `halves_1000` to the counts for `list(range(1000))`, `list(range(1000, 0, -1))` and `list(range(0, 1000, 2)) + list(range(1, 1000, 2))` (work them out with your function).

In [ ]:
from functools import cmp_to_key

sorted_1000 = ___
reversed_1000 = ___
halves_1000 = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert comparisons_of_sorted([]) == 0 and comparisons_of_sorted([2, 1]) == 1\n    assert sorted_1000 == 999 and reversed_1000 == 999\n    assert halves_1000 == comparisons_of_sorted(list(range(0, 1000, 2)) + list(range(1, 1000, 2)))\n    assert halves_1000 < 2100")

### Exercise: Sort by merging runs

Write `natural_merge_sort(items)` that returns a new sorted list by finding the non-decreasing runs (as in the first exercise) and merging neighbouring runs in pairs, round after round, until one is left. It must be stable. On sorted input there's one run, so no merging happens. Don't use `sort` or `sorted`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    for data in ([3, 5, 9, 12, 8, 6, 7, 7, 10, 1], [], [1], list(range(50)), list(range(50, 0, -1))):\n        assert natural_merge_sort(data) == sorted(data)\n    r = random.Random(2)\n    recs = [(r.randint(0, 5), i) for i in range(500)]\n    assert natural_merge_sort(recs) == sorted(recs)\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

### Bug hunt: Unstable reversal

This run finder reverses runs that go down **or stay equal**, and that breaks stability: `[(2, 'a'), (2, 'b'), (1, 'c')]` by first item comes back with `b` before `a`. Fix the falling-run test so it only reverses strictly falling runs. It should print `[(2, 'a'), (2, 'b')] [(1, 'c')]`.

In [ ]:
def first_runs(items, key):
    result, i = [], 0
    while i < len(items):
        j = i + 1
        if j < len(items) and key(items[j]) <= key(items[i]):
            while j < len(items) and key(items[j]) <= key(items[j - 1]):
                j += 1
            result.append(items[i:j][::-1])
        else:
            while j < len(items) and key(items[j]) >= key(items[j - 1]):
                j += 1
            result.append(items[i:j])
        i = j
    return result

print(*first_runs([(2, "a"), (2, "b"), (1, "c")], key=lambda p: p[0]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stable\"\n    expect_output(\"[(2, 'a'), (2, 'b')] [(1, 'c')]\")")

### Bug hunt: A comparison function that lies

This comparison function returns `True` or `False`, but `cmp_to_key` needs a negative number, zero or a positive number, so the sort comes out wrong. Fix `compare` so the program prints `[1, 2, 3, 4]`.

In [ ]:
from functools import cmp_to_key

def compare(a, b):
    return a > b

print(sorted([3, 1, 4, 2], key=cmp_to_key(compare)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sorted\"\n    expect_output(\"[1, 2, 3, 4]\")")

### Puzzle: Counting comparisons

Arrange the lines so the program prints how many comparisons `sorted` makes on `data`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # how many runs does timsort_runs find in [1, 2, 3, 3, 2, 1]? (strictly falling runs are reversed)
b = ___   # comparisons sorted makes on a sorted list of 5,000 items
c = ___   # True or False: Timsort is stable
d = ___   # the Big-O of Timsort in the worst case, as a string like "O(n)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 2 and b == 4999 and c is True and d.replace(\" \", \"\") == \"O(nlogn)\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a mini Timsort

Write `mini_timsort(items, minrun=32)` that sorts a list **in place**, stably, the Timsort way:

1. Walk the list finding runs as Timsort does (reverse strictly falling runs in place).
2. If a run is shorter than `minrun` (and the list isn't finished), extend it to `minrun` items (or to the end) with insertion sort.
3. Keep a stack of `(start, length)` for the runs. After pushing each run, merge the top two runs while the one below the top is no longer than the top one (`stack[-2] length <= stack[-1] length`), which keeps merges balanced.
4. At the end, merge everything left on the stack, top two at a time.

Return the number of merges made. Merging two neighbouring runs means merging `items[a:b]` and `items[b:c]` back into `items[a:c]` (you may make a copy of the left run for that).

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_sorts():\n    \"sorts, stably\"\n    r = random.Random(8)\n    for n in (0, 1, 5, 31, 32, 33, 100, 1000, 5000):\n        data = [r.randint(0, 50) for _ in range(n)]\n        copy = data[:]\n        mini_timsort(copy)\n        assert copy == sorted(data), n\n    recs = [(r.randint(0, 9), i) for i in range(3000)]\n    copy = recs[:]\n    mini_timsort(copy)\n    assert copy == sorted(recs)\ndef test_adaptive():\n    \"adaptive\"\n    data = list(range(10_000))\n    assert mini_timsort(data) == 0 and data == list(range(10_000))\n    data = list(range(10_000, 0, -1))\n    assert mini_timsort(data) == 0\n    data = list(range(0, 10_000, 2)) + list(range(1, 10_000, 2))\n    assert mini_timsort(data) == 1 and data == list(range(10_000))\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

## Warm-up

Python sorted 1,000 already sorted numbers with 999 comparisons: one look at each neighbouring pair, and done. Merge sort would make about 5,000 here, and selection sort about 500,000. Timsort noticed the list was already in order.

## Real data has order in it

The sorts you've written so far treat every input as if it were completely shuffled. Real data rarely is:

- a list that was sorted yesterday, with a few new items added to the end;
- log entries that are in time order apart from a few that arrived late;
- two sorted lists joined together;
- data sorted backwards, or in blocks.

Tim Peters designed Timsort around this observation. On data with existing order it's much faster than O(n log n), down to O(n) on sorted or reversed data. On truly random data it's an ordinary, good O(n log n) merge sort. It's also **stable**, which Python guarantees.

## Runs

A **run** is a stretch of the list that's already in order. Timsort starts by walking through the list and cutting it into runs:

- a run going **up** (each item not smaller than the one before) is used as it is;
- a run going **strictly down** (each item smaller than the one before) is **reversed** in place, which makes it go up. It must be strictly down, so that reversing never swaps equal items, which would break stability.

Finding the runs costs n − 1 comparisons. If the whole list is one run (sorted, or reversed), the sort is finished: **O(n)**.

## Making runs long enough

Random data has very short runs, often only two items long, and merging thousands of tiny runs would waste time. So Timsort sets a **minimum run length**, `minrun`, usually between 32 and 64. Any run shorter than that is extended to `minrun` items by **insertion sort**, which is fast on short lists.

Timsort uses **binary insertion sort**: to place each new item, it finds the spot with binary search (which you know from Week 18) instead of comparing backwards one by one. That cuts comparisons, though the items still have to be shifted along:

`bisect_right` (rather than `bisect_left`) puts the new item **after** any items equal to it, keeping the sort stable.

## Merging runs

Now the list is a sequence of sorted runs, each at least `minrun` long (except perhaps the last). Timsort merges them, like bottom-up merge sort. But it's careful about **which** runs it merges and when:

- It keeps the runs it has found on a **stack** and merges as it goes, always merging **neighbouring** runs (so stability is kept).
- It tries to keep the merges **balanced**, merging runs of similar length. Merging a run of 1,000,000 items with one of 10 is a poor deal: it touches a million items to place ten. The rules it uses make sure every item takes part in only about log₂ n merges.
- When merging, if one run keeps "winning" many times in a row, it switches to **galloping**: it jumps ahead 1, 2, 4, 8, ... places to find how far the winning streak goes, then binary-searches back. That copies whole blocks at once, which is how joining two sorted lists end to end costs only about log n comparisons in the merge instead of n.

You don't need the exact balance rules (they're in a file called `listsort.txt` in the CPython source, worth reading later). A simplified Timsort that finds runs, extends short ones, and merges neighbours in pairs, round after round, already shows all the ideas:

## Measuring sorted

`functools.cmp_to_key` turns a comparison function (returning a negative number, 0 or a positive number) into a key. It's mainly for old-style comparison functions, but it's also a neat way to **count** the comparisons any sort makes:

Sorted and reversed data take n − 1 comparisons. Two sorted halves (interleaved, so the merge has real work to do) take about 2n: n to find the two runs and n for one merge. A sorted list with a few random items added takes little more than n. Random data takes about n log₂ n, just as the theory says it must.

## Practice

## Quiz

## Daily challenge

## Before you go

Find `listsort.txt` in the CPython repository on GitHub (search for "cpython listsort.txt") and read its first few sections: it's Tim Peters' own explanation, with measurements. You'll understand most of it now. Tomorrow: sorting by keys in depth, the way you'll sort things most often in real programs.